# GrieveAI audited MuRIL + LoRA diagnostic

Canonical experiment using checked-in synthetic data only. Metrics are not VCET or real-world performance.

## 1. Fixed configuration
The diagnostic uses seed 42, the duplicate-aware 801/275/247 split, and the same data for the TF-IDF + SVM baseline.

In [ ]:
SEED = 42
BATCH_SIZE = 8
MODEL_NAME = "google/muril-base-cased"
DATA_PATH = "data/processed/grievances_synthetic.csv"
TAXONOMY_PATH = "config/taxonomy.json"
SMOKE_OUTPUT_DIR = "/content/drive/MyDrive/GrieveAI_checkpoints/diagnostic_1_epoch"
MAX_LENGTH = 128


## 2. Clone the pushed repository and record its commit

In [ ]:
!git clone --branch master https://github.com/Virajn07/GrieveAI.git /content/GrieveAI
%cd /content/GrieveAI
from google.colab import drive
drive.mount("/content/drive")
import subprocess
GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Git commit:", GIT_COMMIT)


## 3. Install required ML packages, record versions and check GPU

In [ ]:
!pip install -q -r requirements-colab.txt
import hashlib, platform, torch, transformers, peft, sklearn, pandas as pd
from pathlib import Path
versions = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "transformers": transformers.__version__, "peft": peft.__version__, "sklearn": sklearn.__version__, "pandas": pd.__version__}
manifest_files = ["src/train.py", "src/model.py", "src/data_splits.py", "train_baselines.py", "config/taxonomy.json", DATA_PATH]
source_manifest = {p: hashlib.sha256(Path(p).read_bytes()).hexdigest() for p in manifest_files}
print({"commit": GIT_COMMIT, "versions": versions, "source_hashes": source_manifest})
assert torch.cuda.is_available(), "Select a GPU runtime before the MuRIL check."
assert MODEL_NAME == __import__("src.model", fromlist=["MURIL_CHECKPOINT"]).MURIL_CHECKPOINT


## 4. Validate taxonomy mapping and frozen split

In [ ]:
import json
from src.data_splits import make_splits, validate_taxonomy_labels, has_group_leakage
df = pd.read_csv(DATA_PATH)
taxonomy = json.loads(Path(TAXONOMY_PATH).read_text(encoding="utf-8"))
validate_taxonomy_labels(df, taxonomy)
assert len(df) == 1323 and df.category.nunique() == 7 and df.subcategory.nunique() == 28
assert all(len(v["subcategories"]) == 4 for v in taxonomy["categories"].values())
train_df, val_df, test_df = make_splits(df, seed=SEED)
assert (len(train_df), len(val_df), len(test_df)) == (801, 275, 247)
assert not has_group_leakage((train_df, val_df, test_df))
print({"rows": len(df), "split_sizes": [len(train_df), len(val_df), len(test_df)], "category_counts": df.category.value_counts().sort_index().to_dict(), "taxonomy_mapping": "valid"})


## 5. Run the keyword and TF-IDF + SVM baselines on the same split

In [ ]:
!python train_baselines.py --seed 42


## 6. Test MuRIL tokenizer output, forward pass, hierarchy mask and LoRA

In [ ]:
from transformers import AutoTokenizer
from src.model import GrieveAIClassifier, load_taxonomy, MURIL_CHECKPOINT
model_taxonomy = load_taxonomy(TAXONOMY_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = GrieveAIClassifier(7, 28).to("cuda").eval()
batch = tokenizer("Synthetic example: campus Wi-Fi is unavailable.", return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
with torch.no_grad():
    output = model(**{k: v.to("cuda") for k, v in batch.items()})
assert output["category_logits"].shape == (1, 7)
assert output["subcategory_logits"].shape == (1, 28)
cat = int(output["category_logits"].argmax(-1)[0])
masked = model.mask_subcategory_logits(output["subcategory_logits"][0], cat, model_taxonomy)
allowed = model_taxonomy["category_to_subcat_indices"][model_taxonomy["categories"][cat]]
assert all(torch.isneginf(masked[i]) for i in range(28) if i not in allowed)
print("Forward and parent mask passed; trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


## 7. Run the one-epoch pipeline diagnostic
This checks training, checkpoint writing and reload. Its metrics are diagnostic only.

In [ ]:
!python src/train.py --data data/processed/grievances_synthetic.csv --taxonomy config/taxonomy.json --smoke_test --batch_size 8 --seed 42 --max_length 128 --output_dir /content/drive/MyDrive/GrieveAI_checkpoints/diagnostic_1_epoch


## 8. Inspect diagnostic metrics and explicitly reload the diagnostic checkpoint

In [ ]:
import json
from src.model import MuRILInference
metrics = json.loads((Path(SMOKE_OUTPUT_DIR) / "metrics.json").read_text(encoding="utf-8"))
print(json.dumps({"validation": metrics["validation"], "test": metrics["test"], "split": metrics["split"]}, indent=2))
reloaded = MuRILInference(SMOKE_OUTPUT_DIR, TAXONOMY_PATH, device="cuda")
print(reloaded.predict("Synthetic example: campus Wi-Fi is unavailable."))


## 9. Save the reproducibility manifest

In [ ]:
experiment_manifest = {
    "experiment_name": "muril_lora_synthetic_seed42_diagnostic",
    "git_commit": GIT_COMMIT,
    "dataset_path": DATA_PATH,
    "dataset_sha256": source_manifest[DATA_PATH],
    "taxonomy_path": TAXONOMY_PATH,
    "taxonomy_sha256": source_manifest[TAXONOMY_PATH],
    "random_seed": SEED,
    "split_sizes": [len(train_df), len(val_df), len(test_df)],
    "model_name": MODEL_NAME, "tokenizer_name": MODEL_NAME,
    "lora": {"r": 8, "alpha": 16, "dropout": 0.1, "target_modules": ["query", "value"]},
    "batch_size": BATCH_SIZE, "learning_rate": 2e-4, "max_sequence_length": MAX_LENGTH,
    "optimizer": "AdamW", "scheduler": "none",
    "loss": {"category": "cross_entropy (focal gamma 0)", "subcategory": "gold-parent masked cross_entropy (focal gamma 0)", "priority": "Huber delta 1, weight 0.5"},
    "epochs": 1, "metrics": metrics, "packages": versions, "source_sha256": source_manifest,
    "data_scope": "synthetic only; not VCET performance"
}
manifest_path = Path(SMOKE_OUTPUT_DIR) / "experiment_manifest.json"
manifest_path.write_text(json.dumps(experiment_manifest, indent=2), encoding="utf-8")
print("Saved manifest:", manifest_path)


## Stop here

The controlled three-epoch MuRIL run is intentionally not included. Stop after the mapping checks, baseline, forward check and one-epoch diagnostic. Run three epochs only after a separate explicit instruction.